# 6장 계획 수립

TinyAgent가 추론과 도구 호출을 반복해 작업을 수행합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 6장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [16]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [17]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1. 언어 모델 선택

내장 추론이나 도구 호출이 없는 Gemma 3를 사용합니다. 모델이 한 번 행동한 뒤 결과를 보고 다음 행동을 정하도록 프롬프트와 반복문을 구현합니다.

In [1]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B(내장 추론과 도구 호출 없음)
llm = LLM(model="gemma3:12b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [2]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 여러 단계의 작업 실행

5장에서 연결한 `Tools`를 사용해 `TinyAgent`가 여러 동작을 순서대로 수행하도록 합니다.

![도구 실행 결과를 다음 단계에 전달하는 반복 구조](../images/ch6.png)

어떤 행동을 어떤 순서로 실행할지 결정하는 방법으로 ReAct(Reason and Act)를 구현합니다.

## 3. ReAct의 추론과 행동

에이전트는 과제를 더 작은 작업으로 나누거나 계획을 세워 여러 단계에 걸쳐 처리할 수 있습니다. ReAct에서는 다음 순서를 반복합니다.

- `THOUGHT`에서 모델이 현재 상황과 다음 행동을 생각합니다.
- `ACTION`에서 모델이 도구 호출 등 실행할 동작을 지정합니다.
- `OBSERVATION`에서 프로그램이 실제 실행 결과를 모델에 전달합니다.

`ReAct` 클래스는 모델에 제공할 지침을 만들고, 응답에서 생각과 행동을 분리하고, 도구 결과를 관찰 메시지로 표현합니다. 에이전트는 매 단계에 `ACTION`을 출력합니다. 더 이상 도구가 필요 없으면 `final_answer`를 선택해 답을 전달하고 반복을 끝냅니다.

In [3]:
import re
from illustrated_agents.chapters.ch2 import Response

class ReAct:
    """ReAct의 지침과 응답 해석 방식을 정의한다."""

    def __init__(self, max_steps: int = 10):
        """ReAct를 초기화한다.

        매개변수:
            max_steps: 한 작업에서 실행할 최대 단계 수.
        """
        self.max_steps = max_steps

    @property
    def prompt(self) -> str:
        return """
# ReAct (Reason and Act)

You are a ReAct agent that performs exactly ONE step per turn.

## ReAct Format

You use the following format for each step:

THOUGHT: [Your reasoning about what to do next]
ACTION:
{
    "tool": "a_tool_name",
    "kwargs": {"param": "value"},
}

An observation will be provided after each action. You do not generate the observation yourself.

## ReAct Completion

To provide the final answer to the task, use an action blob with "tool": "final_answer" tool. 
It is the only way to complete the task, else you will be stuck on a loop. 
So your final output should look like this:

ACTION:
{
    "tool": "final_answer",
    "kwargs": "insert your final answer here"
}

Use the `final_answer` tool when you are completely done with all subtasks and have the final answer ready.
You can also use `final_answer` to directly reply to a user's question without using any other tools.

"""

    def parse(self, response: Response) -> Response:
        """ReAct 응답을 THOUGHT와 ACTION으로 분리한다."""
        text = response.content

        # 각 구간을 찾는 패턴
        patterns = {
            "THOUGHT": r"THOUGHT:\s*(.+?)(?=ACTION:|OBSERVATION:|$)",
            "ACTION": r"ACTION:\s*(.+?)(?=THOUGHT:|OBSERVATION:|$)",
        }

        # 정규식으로 각 구간 추출
        result = {}
        for key, pattern in patterns.items():
            match = re.search(pattern, text, re.DOTALL)
            result[key] = match.group(1).strip() if match else ""

        # 응답을 갱신하고 실행할 행동만 추출
        response.content = result["ACTION"]
        response.reasoning = result["THOUGHT"]
        return response

먼저 프롬프트를 읽습니다. 어떤 정보를 보고 다음 행동을 선택할지, 답을 어떤 형식으로 쓸지를 지시합니다. 아래 코드 설명에서 각 지시의 용도를 확인합니다.

프롬프트는 한 차례에 한 단계만 수행하라고 지시합니다. THOUGHT에는 다음 행동을 정한 이유를, ACTION에는 도구 이름과 인자를 씁니다. 관찰값은 프로그램에서 받을 때까지 기다립니다. 모든 하위 작업이 끝나거나 도구 없이 답할 수 있으면 final_answer의 인자로 최종 답변을 냅니다.

In [4]:
from illustrated_agents.chapters.ch6 import react_prompt_annotated; react_prompt_annotated

응답에서 `THOUGHT`와 `ACTION`을 찾아 서로 분리합니다.

In [5]:
from illustrated_agents.chapters.ch6 import react_parse_annotated; react_parse_annotated

도구 실행이나 최종 답변 반환에 사용하는 부분은 `ACTION`입니다. `THOUGHT`는 모델이 선택한 행동을 해석할 때 참고할 수 있습니다.

이 예제의 정규식은 모델이 `THOUGHT:`를 `THOUGH:`로 잘못 쓰면 해당 부분을 찾지 못합니다. 간단한 파서가 출력 형식에 얼마나 의존하는지 확인할 수 있습니다. 도구 호출용 특수 토큰을 학습한 모델은 이런 형식 오류를 줄일 수 있습니다. [내장 ReAct 실습](chapter06_native_react.ipynb)에서 두 구현을 비교합니다.

## 4. `agent.py` 수정

- `__init__`에서 시스템 메시지에 ReAct 지침을 추가합니다.
- `run`에서 `ReAct`에 지정한 최대 단계 수만큼 반복합니다.
- `_step`에서 생각, 행동, 관찰로 구성된 한 단계를 처리합니다.
- `_execute_action`에서 행동에 지정된 도구 호출을 처리합니다.

In [6]:
from illustrated_agents.chapters.ch2 import Response, Trajectory
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import Tools


class TinyAgent:
    """교육용 에이전트의 기본 클래스를 정의한다."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools, planner: ReAct):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner

        self.trajectory = Trajectory()

        # 시스템 프롬프트에 각 구성 요소의 지침을 추가
        system_prompt = "You are a helpful assistant.\n\n"
        system_prompt += self.planner.prompt
        system_prompt += self.tools.prompt
        self.memory.add("system", system_prompt)

    def run(self, task: str) -> str:
        """주어진 작업으로 에이전트를 실행한다."""
        self.memory.add("user", task)
        self.trajectory.initialize(task)

        # 여러 단계를 수행하는 반복문
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

    def _step(self) -> str | None:
        """에이전트의 한 단계를 실행한다."""
        # THOUGHT: 응답을 생성해 메모리에 추가
        response = self.llm.generate(
            self.memory.get_messages(), tools=self.tools.schemas
        )
        self.memory.add(
            "assistant", response.content, tool_call=response.tool_call
        )

        # 도구 호출 해석
        response = self.planner.parse(response)
        response = self.tools.parse(response)

        # ANSWER: 종료 조건 확인
        if self.tools.is_done(response):
            self.trajectory.add(response)
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """도구가 수행할 동작을 실행한다."""

        # ACTION: 도구 실행
        result = self.tools.execute(response)

        # OBSERVATION: 도구 결과를 메모리에 추가하고 화면에 표시
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)
        self.trajectory.add(response, observation)

        return None


먼저 `__init__`에서 구성 요소를 연결하고 시스템 프롬프트를 만드는 부분을 확인합니다.

In [7]:
from illustrated_agents.chapters.ch6 import tinyagent_init_annotated; tinyagent_init_annotated

다음은 `run`입니다. 모델이 답을 완성했을 때와 최대 단계 수에 도달했을 때의 처리를 살펴봅니다.

In [8]:
from illustrated_agents.chapters.ch6 import tinyagent_run_annotated; tinyagent_run_annotated

`_step`이 한 차례의 모델 응답을 처리하고 행동을 선택하는 순서를 확인합니다.

In [9]:
from illustrated_agents.chapters.ch6 import tinyagent_step_annotated; tinyagent_step_annotated

`_execute_action`에서는 `ACTION`에 들어 있는 요청을 실행합니다.

In [10]:
from illustrated_agents.chapters.ch6 import tinyagent_action_annotated; tinyagent_action_annotated

이전 장의 코드와 전체 변경 사항을 비교합니다. 여러 단계의 처리를 위해 `for` 반복문이 추가되었습니다.

In [11]:
from illustrated_agents.chapters.ch6 import tinyagents_diff; tinyagents_diff

이제 구성 요소를 연결해 에이전트가 여러 동작을 스스로 선택할 수 있는지 확인합니다.

## 5. 여러 단계를 수행하는 `TinyAgent` 실행

(4.6 + 6.685)에 4를 곱하고 그 결과에서 3.14를 빼 달라고 요청합니다.

In [12]:
from illustrated_agents.toolbox import add, multiply, subtract

# 도구 등록
tools = Tools()
tools.add_tool("add", add, "add(a: str, b: str)")
tools.add_tool("subtract", subtract, "subtract(a: str, b: str)")
tools.add_tool("multiply", multiply, "multiply(a: str, b: str)")

# 메모리
memory = Memory()

# ReAct
react = ReAct(max_steps=10)

# 에이전트 생성
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react)

# 여러 단계의 추론이 필요한 작업
agent.run("What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?")

'42.0'

저자의 Gemma 3 12B 실행에서는 요청한 작업을 차례로 수행하고 최종 답변을 반환했습니다. 아래에서 실제로 몇 단계를 거쳤는지 확인합니다.

In [13]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

In [14]:
from rich import print

print(agent.trajectory.runs)

[
    {
        'query': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?',
        'steps': [
            Step(
                thought='First, I need to add 4.6 and 6.685. Then I need to multiply the sum by 4. Finally, I need 
to subtract 3.14 from the result.',
                action={'tool': 'add', 'kwargs': {'a': '4.6', 'b': '6.685'}},
                observation='OBSERVATION: 11.285',
                answer=None,
                metadata=None
            ),
            Step(
                thought='Now I need to multiply 11.285 by 4.',
                action={'tool': 'multiply', 'kwargs': {'a': '11.285', 'b': '4'}},
                observation='OBSERVATION: 45.14',
                answer=None,
                metadata=None
            ),
            Step(
                thought='Now I need to subtract 3.14 from 45.14.',
                action={'tool': 'subtract', 'kwargs': {'a': '45.14', 'b': '3.14'}},
                observation='OBSERVATION: 42.0',
                answer=None,
                metadata=None
            ),
            Step(
                thought='I have completed all the steps. Now I can provide the final answer.',
                action=None,
                observation=None,
                answer='42.0',
                metadata=None
            )
        ]
    }
]

저자의 예에서는 도구를 차례로 세 번 호출한 뒤 결과를 답변에 담았습니다. 대화 메모리를 열어 어떤 메시지가 전달되었는지 확인합니다.

In [15]:
from rich import print
print(agent.memory.messages)

[
    {
        'role': 'system',
        'content': 'You are a helpful assistant.\n\n\n# ReAct (Reason and Act)\n\nYou are a ReAct agent that 
performs exactly ONE step per turn.\n\n## ReAct Format\n\nYou use the following format for each step:\n\nTHOUGHT: 
[Your reasoning about what to do next]\nACTION:\n{\n    "tool": "a_tool_name",\n    "kwargs": {"param": 
"value"},\n}\n\nAn observation will be provided after each action. You do not generate the observation 
yourself.\n\n## ReAct Completion\n\nTo provide the final answer to the task, use an action blob with "tool": 
"final_answer" tool. \nIt is the only way to complete the task, else you will be stuck on a loop. \nSo your final 
output should look like this:\n\nACTION:\n{\n    "tool": "final_answer",\n    "kwargs": "insert your final answer 
here"\n}\n\nUse the `final_answer` tool when you are completely done with all subtasks and have the final answer 
ready.\nYou can also use `final_answer` to directly reply to a user\'s question without using any other 
tools.\n\n\n# Tools\n\nIf needed, you can only use the following tools to assist you in completing tasks:\n\n`add`:
add(a: str, b: str)\n`subtract`: subtract(a: str, b: str)\n`multiply`: multiply(a: str, b: str)\n\nTo use a tool, 
respond with JSON: {"tool": "name", "kwargs": {"param": "value"}}\n'
    },
    {'role': 'user', 'content': 'What is (4.6 + 6.685) x 4, and then subtract 3.14 from the result?'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: First, I need to add 4.6 and 6.685. Then I need to multiply the sum by 4. Finally, I 
need to subtract 3.14 from the result.\nACTION:\n{\n    "tool": "add",\n    "kwargs": {"a": "4.6", "b": 
"6.685"}\n}\n'
    },
    {'role': 'user', 'content': 'OBSERVATION: 11.285'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: Now I need to multiply 11.285 by 4.\nACTION:\n{\n    "tool": "multiply",\n    
"kwargs": {"a": "11.285", "b": "4"}\n}'
    },
    {'role': 'user', 'content': 'OBSERVATION: 45.14'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: Now I need to subtract 3.14 from 45.14.\nACTION:\n{\n    "tool": "subtract",\n    
"kwargs": {"a": "45.14", "b": "3.14"}\n}'
    },
    {'role': 'user', 'content': 'OBSERVATION: 42.0'},
    {
        'role': 'assistant',
        'content': 'THOUGHT: I have completed all the steps. Now I can provide the final answer.\nACTION:\n{\n    
"tool": "final_answer",\n    "kwargs": "42.0"\n}'
    }
]

---

# 구현한 내용

`for` 반복문과 ReAct를 결합해 에이전트가 앞 단계의 결과를 보고 다음 행동을 결정하게 했습니다. 형식을 직접 지정하는 과정을 통해 추론과 행동, 도구의 반환값이 어떻게 연결되는지 확인했습니다.

In [16]:
from illustrated_agents.chapters.ch6 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py      ← Updated (Autonomy with a for-loop and ReAct planner.)                                       │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py   ← New (Added the ReAct (Reason and Act) framework)                                            │
│ ├── toolbox.py                                                                                                  │
│ ├── tools.py                                                                                                    │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# 다음 실습

[내장 ReAct](chapter06_native_react.ipynb)에서는 같은 반복문에 모델 자체의 추론과 도구 호출을 연결합니다.